In [ ]:
import re
from collections import defaultdict, Counter
import emoji

# LLM instruction set

You are assisting with a Python tool that analyzes WhatsApp chat exports (iOS format).
Dependencies (reuse when it makes sense): re, collections (Counter, defaultdict), pathlib, emoji
Parsed message object structure:
```
{
  "date": "DD/MM/YYYY",
  "time": "HH.MM.SS",
  "sender": "Sender name (cleaned)",
  "msg": "Full message text (may contain newlines)"
}
```
All messages are stored in a list: `messages = [ {...}, {...}, ... ]`

In [ ]:
import re

line_re = re.compile(
    r"^\[(?P<date>\d{2}/\d{2}/\d{4}), (?P<time>\d{2}\.\d{2}\.\d{2})\] (?P<sender>.*?): (?P<msg>.*)$"
)

messages = []
current = None

with open("/Users/mikaelthorup/_chat.txt", encoding="utf-8") as f:
    for raw in f:
        line = raw.rstrip("\n")

        m = line_re.match(line)
        if m:
            # starting a new message → push previous
            if current:
                messages.append(current)

            current = {
                "date": m.group("date"),
                "time": m.group("time"),
                "sender": m.group("sender"),
                "msg": m.group("msg"),
            }
        else:
            # continuation → append to last message
            if current:
                current["msg"] += "\n" + line

# add last message
if current:
    messages.append(current)

In [ ]:
# debug
for m in messages[0:100]:
    print(f"{m['date']} {m['time']} | {m['sender']}: {m['msg']}")

# Messages

In [ ]:
# Count messages per person
msg_count = Counter(m["sender"] for m in messages)
msg_count

In [ ]:
# emojis per person
emoji_counts = defaultdict(Counter)

for m in messages:
    text = m["msg"]
    for ch in text:
        if ch in emoji.EMOJI_DATA:
            emoji_counts[m["sender"]][ch] += 1

emoji_counts

In [ ]:
# Average message length
avg_length = {}
for sender in msg_count:
    lengths = [len(m["msg"]) for m in messages if m["sender"] == sender]
    avg_length[sender] = sum(lengths) / len(lengths)